In [ ]:
import gc
import json
import sqlite3
import uuid
from pathlib import Path
from collections import defaultdict

from qdrant_client import QdrantClient, models
from tuutrag.data import D

In [ ]:
DB_PATH: Path = "../../tuutrag.db"

QDRANT_HOST: str      = "localhost"
QDRANT_PORT: int      = 6333
EMBEDDING_DIM: int    = 3072
DISTANCE: models.Distance = models.Distance.COSINE

BRANCH_COLLECTION: str = "branches"
LEAF_COLLECTION: str   = "leafs"

UPSERT_BATCH: int = 256
SQL_FETCH: int = UPSERT_BATCH

In [ ]:
qclient = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=120, https=False)
print(f"Connected to Qdrant at http://{QDRANT_HOST}:{QDRANT_PORT}/dashboard")

In [ ]:
for name in (BRANCH_COLLECTION, LEAF_COLLECTION):
    if qclient.collection_exists(name):
        qclient.delete_collection(name)

    qclient.create_collection(
        collection_name=name,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIM,
            distance=DISTANCE,
        ),
    )
    print(f"Collection '{name}' created ({EMBEDDING_DIM}-dim, {DISTANCE.name})")

In [ ]:
qclient.create_payload_index(BRANCH_COLLECTION, "artifact_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(BRANCH_COLLECTION, "type",
                             field_schema=models.PayloadSchemaType.KEYWORD)

qclient.create_payload_index(LEAF_COLLECTION, "branch_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(LEAF_COLLECTION, "artifact_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)

print("Payload indexes created.")

In [ ]:
def qdrant_point_id(text_uuid: str) -> str:
    """Deterministic point ID from the text UUID."""
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, text_uuid))

def iter_rows(cursor: sqlite3.Cursor, size: int = SQL_FETCH):
    """Yield rows in pages from a cursor."""
    while True:
        rows = cursor.fetchmany(size)
        if not rows:
            break
        yield rows

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

cur = conn.cursor()
cur.execute("""
    SELECT
        be.branch_uuid,
        be.embedding,
        b.artifact_uuid,
        b.chunk,
        b.path,
        a.type
    FROM branch_embeddings  be
    JOIN branches           b  ON b.uuid = be.branch_uuid
    JOIN artifacts          a  ON a.uuid = b.artifact_uuid
""")

branch_count = 0

for page in iter_rows(cur):
    points = []
    for row in page:
        vec = json.loads(row["embedding"])
        points.append(
            models.PointStruct(
                id=qdrant_point_id(row["branch_uuid"]),
                vector=vec,
                payload={
                    "uuid":          row["branch_uuid"],
                    "artifact_uuid": row["artifact_uuid"],
                    "type":          row["type"],
                    "chunk":         row["chunk"],
                    "path":          row["path"],
                },
            )
        )
    qclient.upsert(collection_name=BRANCH_COLLECTION, points=points, wait=True)
    branch_count += len(points)
    del points, page
    gc.collect()

    if branch_count % (UPSERT_BATCH * 10) == 0:
        print(f"  branches upserted: {branch_count:,}")

cur.close()
print(f"Branch upsert complete — {branch_count:,} points")

In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT
        le.leaf_uuid,
        le.embedding,
        l.branch_uuid,
        l.text,
        b.artifact_uuid,
        b.path,
        a.type
    FROM leaf_embeddings  le
    JOIN leafs            l  ON l.uuid        = le.leaf_uuid
    JOIN branches         b  ON b.uuid        = l.branch_uuid
    JOIN artifacts        a  ON a.uuid        = b.artifact_uuid
""")

leaf_count = 0

for page in iter_rows(cur):
    points = []
    for row in page:
        vec = json.loads(row["embedding"])
        points.append(
            models.PointStruct(
                id=qdrant_point_id(row["leaf_uuid"]),
                vector=vec,
                payload={
                    "uuid":          row["leaf_uuid"],
                    "branch_uuid":   row["branch_uuid"],
                    "artifact_uuid": row["artifact_uuid"],
                    "type":          row["type"],
                    "text":          row["text"],
                    "path":          row["path"],
                },
            )
        )
    qclient.upsert(collection_name=LEAF_COLLECTION, points=points, wait=True)
    leaf_count += len(points)
    del points, page
    gc.collect()

    if leaf_count % (UPSERT_BATCH * 10) == 0:
        print(f"  leafs upserted: {leaf_count:,}")

cur.close()
conn.close()
print(f"Leaf upsert complete — {leaf_count:,} points")

In [ ]:
for name in (BRANCH_COLLECTION, LEAF_COLLECTION):
    info = qclient.get_collection(name)
    exact = qclient.count(collection_name=name, exact=True).count
    print(f"\n{name}:")
    print(f"  points:       {info.points_count:,}")
    print(f"  exact count:  {exact:,}")
    print(f"  index status: {info.status}")

In [ ]:
def search_similar_branches(
    branch_uuid: str,
    k: int = 10,
    type_filter: str | None = None,
) -> list[models.ScoredPoint]:
    """ Find top-k branches most similar to branch_uuid. """
    point_id = qdrant_point_id(branch_uuid)

    query_filter = None
    if type_filter:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="type",
                    match=models.MatchValue(value=type_filter),
                )
            ]
        )

    return qclient.query_points(
        collection_name=BRANCH_COLLECTION,
        query=point_id,
        using=None,
        limit=k,
        query_filter=query_filter,
        with_payload=True,
    ).points


def search_similar_leafs(
    leaf_uuid: str,
    k: int = 10,
    artifact_uuid_filter: str | None = None,
    branch_uuid_filter: str | None = None,
) -> list[models.ScoredPoint]:
    """ Find top-k leafs most similar to leaf_uuid. """
    point_id = qdrant_point_id(leaf_uuid)

    conditions = []
    if artifact_uuid_filter:
        conditions.append(
            models.FieldCondition(
                key="artifact_uuid",
                match=models.MatchValue(value=artifact_uuid_filter),
            )
        )
    if branch_uuid_filter:
        conditions.append(
            models.FieldCondition(
                key="branch_uuid",
                match=models.MatchValue(value=branch_uuid_filter),
            )
        )

    query_filter = models.Filter(must=conditions) if conditions else None

    return qclient.query_points(
        collection_name=LEAF_COLLECTION,
        query=point_id,
        using=None,
        limit=k,
        query_filter=query_filter,
        with_payload=True,
    ).points

In [ ]:
results = search_similar_branches("041f16b3-f3b7-488f-a27c-074e63a9d300.1", k=3)
for r in results:
    print(f"  {r.score:.4f}  {r.payload['uuid']}  {r.payload['type']}    {r.payload['chunk']}")

In [ ]:
results = search_similar_leafs(
    "041f16b3-f3b7-488f-a27c-074e63a9d300.1.1",
    k=5,
    artifact_uuid_filter="041f16b3-f3b7-488f-a27c-074e63a9d300"
)
for r in results:
    print(f"  {r.score:.4f}  {r.payload['uuid']}  {r.payload['branch_uuid']}")

In [ ]:
def artifact_uuid_from_branch(branch_uuid: str) -> str:
    """'aaaa-bbbb.3' → 'aaaa-bbbb'"""
    return branch_uuid.rsplit(".", 1)[0]

In [ ]:
# Scroll through every point in the branches collection and group by artifact_uuid
artifact_branches: dict[str, list[str]] = defaultdict(list)

offset = None
page_size = 5000
total_scrolled = 0

while True:
    records, next_offset = qclient.scroll(
        collection_name=BRANCH_COLLECTION,
        limit=page_size,
        offset=offset,
        with_payload=["uuid", "artifact_uuid"],
        with_vectors=False,
    )
    for record in records:
        b_uuid = record.payload["uuid"]
        a_uuid = record.payload["artifact_uuid"]
        artifact_branches[a_uuid].append(b_uuid)
    total_scrolled += len(records)

    if next_offset is None:
        break
    offset = next_offset

print(f"Artifacts:       {len(artifact_branches):,}")
print(f"Total branches:  {total_scrolled:,}")

In [ ]:
entity_conn = sqlite3.connect(str(DB_PATH))
entity_conn.row_factory = sqlite3.Row


def get_entities_for_branches(branch_uuids: list[str]) -> list[str]:
    """ Get all leafs from a given branch and combine enities"""
    if not branch_uuids:
        return []

    placeholders = ",".join("?" for _ in branch_uuids)
    cur = entity_conn.cursor()
    cur.execute(
        f"""
        SELECT b.uuid AS branch_uuid, l.entities
        FROM   branches b
        LEFT JOIN leafs l ON b.uuid = l.branch_uuid
        WHERE  b.uuid IN ({placeholders})
        """,
        branch_uuids,
    )

    combined: list[str] = []
    for row in cur:
        raw = row["entities"]
        if raw:
            parsed: list[str] = json.loads(raw)
            combined.extend(parsed)
    cur.close()

    return combined

In [ ]:
output: list[dict] = []
processed = 0
total_branches = sum(len(v) for v in artifact_branches.values())

for a_uuid, branches_in_artifact in artifact_branches.items():

    for branch_uuid in branches_in_artifact:
        # Filter Qdrant: only return branches from the SAME artifact
        point_id = qdrant_point_id(branch_uuid)
        results = qclient.query_points(
            collection_name=BRANCH_COLLECTION,
            query=point_id,
            using=None,
            limit=3,
            query_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="artifact_uuid",
                        match=models.MatchValue(value=a_uuid),
                    )
                ]
            ),
            with_payload=True,
        ).points

        # Get the current branch's chunk from Qdrant
        current_point = qclient.retrieve(
            collection_name=BRANCH_COLLECTION,
            ids=[qdrant_point_id(branch_uuid)],
            with_payload=True,
            with_vectors=False,
        )
        current_chunk = current_point[0].payload["chunk"] if current_point else ""

        # Collect the current branch + 3 similar branch 
        sim_uuids = [r.payload["uuid"] for r in results]
        all_uuids = [branch_uuid] + sim_uuids

        # Get combined entities
        combined_entities = get_entities_for_branches(all_uuids)

        # sim_branches: current branch first, then the 3 similar
        sim_branches = [{"uuid": branch_uuid, "chunk": current_chunk}]
        sim_branches.extend(
            {"uuid": r.payload["uuid"], "chunk": r.payload["chunk"]}
            for r in results
        )

        record = {
            "uuid": branch_uuid,
            "entities": combined_entities,
            "sim_branches": sim_branches,
        }
        output.append(record)

        processed += 1
        if processed % 1_000 == 0:
            print(f"  processed {processed:,} / {total_branches:,} branches")

print(f"\nDone — {processed:,} branch records built, "
      f"with {sum(len(r['sim_branches']) for r in output):,} total similarity entries")

In [ ]:
OUTPUT_FILE = "../data/processed/vector_sim_branches.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

import os
file_size = os.path.getsize(OUTPUT_FILE)
print(f"Written to: {os.path.abspath(OUTPUT_FILE)}")
print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")
print(f"Records:    {len(output):,}")

entity_conn.close()

In [ ]:
def append_artifact_summary(data: list[dict], summaries_path: str, output_file: str = "vector_sim_branches.json") -> list[dict]:
    """ For every record in data, look up the artifact summary using branch uuid and parsing it."""

    with open(summaries_path, "r") as f:
        summaries_list: list[dict] = json.load(f)

    # artifact_uuid → summary 
    summary_map: dict[str, str] = {
        item["uuid"]: item["text"]
        for item in summaries_list
    }
    print(f"Loaded {len(summary_map):,} artifact summaries from {summaries_path}")

    # Extract artifact_uuid: "aaaa-bbbb.3" → "aaaa-bbbb"
    def artifact_uuid_from_branch(branch_uuid: str) -> str:
        return branch_uuid.rsplit(".", 1)[0]

    matched = 0
    for record in data:
        a_uuid = artifact_uuid_from_branch(record["uuid"])
        summary = summary_map.get(a_uuid, "")
        record["artifact_summary"] = summary
        if summary:
            matched += 1

    reordered: list[dict] = []
    for record in data:
        reordered.append({
            "uuid": record["uuid"],
            "entities": record["entities"],
            "artifact_summary": record["artifact_summary"],
            "sim_branches": record["sim_branches"],
        })

    print(f"Matched summaries:  {matched:,} / {len(reordered):,} records")

    with open(output_file, "w") as f:
        json.dump(reordered, f, indent=2)

    file_size = os.path.getsize(output_file)
    print(f"Written to: {os.path.abspath(output_file)}")
    print(f"File size:  {file_size / (1024 * 1024):.1f} MB  ({file_size:,} bytes)")
    print(f"Records:    {len(reordered):,}")

    return reordered


SUMMARIES_PATH = "../data/processed/artifact_summaries.json"
output = append_artifact_summary(output, SUMMARIES_PATH)